# 归档材料

本节的简化评测不能用来验证第 05 章模型改善了驾驶行为；模型输出尚未进入此处的执行链。

[归档索引](../README.md) · [当前学习入口](../../../course/first_loop/README.md)

# 08 · Data & Evaluation：从 sensor bundle 到 failure slice

本章合并旧版 `10`、`17`、`11` 和 `12`。它把“一个 demo”变成可回归的开发流程：

```text
sensor bundle → scenario ID → replay → open/closed-loop metrics → slice → mining
```

评测至少要同时报告安全、任务、舒适性、数据质量和 runtime；平均分不能掩盖 `night/occlusion`、`cut-in` 或 `GNSS outage`。

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents)
                    if (path / "src" / "ad_tutorial").is_dir())
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from ad_tutorial import (
    ARTIFACT_DIR,
    BEVConfig,
    build_bev_dataset,
    build_urban_cut_in_scene,
    ensure_artifact_dir,
    load_json_artifact,
    load_numpy_artifact,
    save_json_artifact,
    save_numpy_artifact,
    scene_to_bev,
)

ensure_artifact_dir()
print("project root:", PROJECT_ROOT)
print("artifact directory:", ARTIFACT_DIR)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import zlib

prediction = load_numpy_artifact("06_prediction.npz")
planner = load_numpy_artifact("07_planner.npz")

def make_bundle(seed, lidar_age_s=0.03, camera_age_s=0.04, sensor_fault="none"):
    scene = build_urban_cut_in_scene(seed=seed, timestamp_s=0.0,
                                     sensor_lag_s=max(camera_age_s - lidar_age_s, 0.0),
                                     scene_id=f"cut_in_{seed:03d}")
    return {
        "scene_id": scene.scene_id,
        "scenario": "urban_cut_in",
        "camera": scene.camera_points,
        "lidar": scene.lidar_points,
        "camera_age_s": camera_age_s,
        "lidar_age_s": lidar_age_s,
        "sensor_fault": sensor_fault,
    }

bundles = [make_bundle(seed) for seed in range(20, 28)]
print("bundle schema:", sorted(k for k in bundles[0] if k not in {"camera", "lidar"}))

In [ ]:
def replay_bundle(bundle, guarded=True):
    rng = np.random.default_rng(zlib.crc32(bundle["scene_id"].encode("utf-8")) % 10000)
    gap = 18.0
    ego_speed = 8.0
    rows = []
    for step in range(60):
        stale = bundle["camera_age_s"] > 0.15 or bundle["lidar_age_s"] > 0.10
        faulty = bundle["sensor_fault"] != "none"
        risk = gap < 10.0 or stale or faulty
        target_speed = 8.0 if (not guarded or not risk) else 2.5
        acceleration = np.clip((target_speed - ego_speed) * 1.5, -4.0, 2.0)
        ego_speed = max(0.0, ego_speed + acceleration * 0.1)
        gap += (3.0 - ego_speed) * 0.1
        rows.append({"gap_m": gap, "speed": ego_speed, "acceleration": acceleration,
                     "latency_ms": 65 + rng.lognormal(1.0, 0.15), "risk": risk})
    frame = pd.DataFrame(rows)
    jerk = np.diff(frame.acceleration, prepend=frame.acceleration.iloc[0]) / 0.1
    return {
        "scene_id": bundle["scene_id"], "collision": bool((frame.gap_m < 2.0).any()),
        "min_gap_m": float(frame.gap_m.min()), "progress_m": float((frame.speed * 0.1).sum()),
        "max_jerk_mps3": float(np.abs(jerk).max()),
        "p95_latency_ms": float(frame.latency_ms.quantile(0.95)),
        "sensor_fault": bundle["sensor_fault"], "stale": bool(frame.risk.iloc[0]),
    }

reports = []
for bundle in bundles:
    reports.append(replay_bundle(bundle, guarded=True))
    if bundle["scene_id"].endswith("022"):
        bundle["sensor_fault"] = "lidar_dropout"
report = pd.DataFrame(reports)
report["slice"] = np.where(report.sensor_fault != "none", "sensor_fault", "nominal")
display(report.head())
display(report.groupby("slice").agg(episodes=("scene_id", "count"), collision_rate=("collision", "mean"),
                                     min_gap_m=("min_gap_m", "mean"), p95_latency_ms=("p95_latency_ms", "max")))

## Real-data checkpoint: replay a fixed nuScenes mini sample

The course's default execution stays offline and synthetic. For a credible portfolio, run the same bundle schema on a fixed nuScenes mini sample and record dataset version, sample token, coordinate convention, time policy and the exact command. The important lesson is the **format of evidence**, not a fabricated benchmark number.

In [ ]:
import os
real_checkpoint = {
    "dataset": "nuScenes mini (optional)",
    "sample_token": os.environ.get("NUSCENES_SAMPLE_TOKEN", "not supplied"),
    "status": "not run" if not os.environ.get("NUSCENES_ROOT") else "ready for fixed-sample replay",
    "required_record": ["dataset version", "sample token", "frame chain", "timestamp policy", "split"],
}
if os.environ.get("NUSCENES_ROOT"):
    try:
        from nuscenes.nuscenes import NuScenes
        nusc = NuScenes(version="v1.0-mini", dataroot=os.environ["NUSCENES_ROOT"], verbose=False)
        sample = nusc.sample[0] if real_checkpoint["sample_token"] == "not supplied" else nusc.get("sample", real_checkpoint["sample_token"])
        real_checkpoint.update({
            "sample_token": sample["token"],
            "lidar_file": nusc.get("sample_data", sample["data"]["LIDAR_TOP"])["filename"],
            "camera_file": nusc.get("sample_data", sample["data"]["CAM_FRONT"])["filename"],
            "status": "sample metadata loaded; run fixed replay and save output",
        })
    except ImportError as exc:
        real_checkpoint.update({"status": "dependency missing", "error": str(exc)})
print(real_checkpoint)

In [ ]:
worst = report.sort_values(["collision", "p95_latency_ms"], ascending=[False, False]).head(3)
print("corner-case candidates:")
display(worst[["scene_id", "collision", "min_gap_m", "p95_latency_ms", "sensor_fault"]])
save_json_artifact("08_eval_report.json", {
    "episodes": int(len(report)),
    "collision_rate": float(report.collision.mean()),
    "mean_min_gap_m": float(report.min_gap_m.mean()),
    "max_p95_latency_ms": float(report.p95_latency_ms.max()),
    "slice_counts": report["slice"].value_counts().to_dict(),
    "real_data_checkpoint": real_checkpoint,
    "next": "09_safety_runtime.ipynb",
})
print("saved evaluation artifact")

### 完成标准

提交一个固定 scenario matrix、bundle schema、open/closed-loop 指标、slice breakdown 和一次 failure replay。至少做两个 ablation（例如 sensor age、LiDAR dropout、planner guard），并记录为什么平均 composite score 不能代替安全门禁。